# Predicting Next-Day Rain in Rwanda: Traditional Machine Learning versus Deep Learning

* **GitHub repository:** [https://github.com/Flaviaa4/rwanda-rain-prediction](https://github.com/Flaviaa4/rwanda-rain-prediction)
* **Google Drive folder (raw data and saved models):** *add link here*
* **Video presentation:** *add link here*

Most agriculture in Rwanda is rain-fed, and climate change is making the timing of the
rainy seasons less predictable. This notebook asks whether today's weather at a location
can predict whether it will rain there tomorrow, to support planting, spraying and
harvesting decisions. The task is framed as binary classification on 42 years of daily
NASA POWER data for 10 locations, and is solved with both scikit-learn models and
TensorFlow/Keras networks fed through `tf.data`, compared on identical data, features,
splits and metrics. All code lives in the `src/` package; this notebook calls it and
displays the results.

## 1. Environment setup

**What.** Detects Colab, mounts Google Drive, clones (or updates) the repository,
installs the requirements and makes `src` importable.
**Why.** Code is versioned on GitHub while data, models and results live on Drive, so
the notebook can be re-run from a fresh runtime without losing finished experiments.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Flaviaa4/rwanda-rain-prediction.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Drive holds the data, models and results so that they survive runtime disconnects.
    from google.colab import drive
    drive.mount("/content/drive")

    # The code always comes from GitHub: clone it the first time, update it afterwards.
    # A folder without src/ is a leftover clone of an empty repository, so it is replaced.
    REPO_DIR = Path("/content/rwanda-rain-prediction")
    if (REPO_DIR / "src").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    requirements = REPO_DIR / "requirements.txt"
    if not requirements.exists():
        raise FileNotFoundError(f"{requirements} is missing: push the code to {REPO_URL} first.")

    # pip's messages are captured and shown only on failure, so the cell stays quiet when
    # Colab already provides every package.
    install = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                              str(requirements)], capture_output=True, text=True)
    if install.returncode:
        print(install.stdout[-3000:])
        print(install.stderr[-3000:])
        raise RuntimeError("pip install failed; see the messages above.")
else:
    # Locally the notebook is opened from the notebooks/ folder of the repository.
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)

**What.** Imports the pipeline modules, loads both configuration files, resolves the
project folders and fixes the random seed.
**Why.** Every setting (paths, dates, thresholds, feature windows, hyperparameters) comes
from `configs/`, so the notebook contains no magic numbers. The `check` helper compares
each sanity checkpoint with the values observed when the pipeline was first validated
and prints `MISMATCH` if anything differs.

In [ ]:
import json

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from tensorflow import keras

from src import plots
from src.baselines import predict_baseline
from src.config import load_config, load_experiments, resolve_paths, set_global_seed
from src.data_prep import add_target, clean, load_raw
from src.dl import DLData
from src.evaluation import (error_analysis, final_test_evaluation, dl_permutation_importance,
                            logreg_coefficients, ml_learning_curve, paradigm_comparison,
                            rank_runs, save_final_models, select_best)
from src.features import build_features, target_correlations
from src.metrics import ResultsLog, compute_metrics
from src.ml import load_best_params
from src.runner import results_path, run_experiments
from src.split import split_by_year

cfg = load_config()
experiments = load_experiments()
paths = resolve_paths(cfg)
SEED = cfg["project"]["seed"]
TARGET = cfg["target"]["name"]
FIGURES = paths.figures_dir
set_global_seed(SEED)
pd.set_option("display.max_columns", 60)
# Permutation importance calls the network many times; TensorFlow's retracing notices
# are harmless there, so only errors are shown.
tf.get_logger().setLevel("ERROR")


def check(label, observed, expected, tolerance=0.0):
    """Print whether an observed value matches the validated reference value."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(observed - expected) <= tolerance
    else:
        ok = observed == expected
    print(f"{'OK      ' if ok else 'MISMATCH'} {label}: {observed} (expected {expected})")


print("TensorFlow", tf.__version__, "| project folder:", paths.project_dir,
      "| run mode:", cfg["run"]["mode"])

## 2. Problem statement and dataset

**Question.** Given everything observable at a location by the end of today, will
tomorrow be a rain day there?

**Target.** `RainTomorrow = 1` when tomorrow's corrected precipitation (`PRECTOTCORR`) at
the same location is at least 1.0 mm, otherwise 0. The 1 mm threshold is a common
agronomic definition of a rain day that ignores trace amounts.

**Data.** Daily NASA POWER values, 1981 to 2025, for 12 district centres. Two centres
share a NASA grid cell with another centre and are removed, leaving 10 locations
(notebook `01_download_data.ipynb` explains why). Fourteen parameters describe rainfall,
temperature, humidity, pressure, wind, radiation, cloud and soil moisture.

**Why it is hard.** Rain tomorrow depends on weather systems that may not yet be visible
at the location today, so the achievable accuracy is limited. A simple rule ("tomorrow
will be like today") is a strong reference that any model must beat to be useful.

## 3. Loading and cleaning

**What.** `load_raw` reads the combined raw table built by notebook 01. `clean` then
(1) replaces NASA's missing-value code with `NaN`, (2) re-checks duplicate grid cells,
(3) keeps dates from 1984, (4) verifies that every location has a continuous daily
sequence and (5) verifies that no missing values remain.
**Why.** Solar radiation and cloud cover are unavailable before 1984; starting there
removes every missing value without imputing anything. The continuity check protects
the lag features, which assume that consecutive rows are consecutive days.
**How to read.** The first table lists each step with the rows before and after; the
second shows the columns that had missing values.

In [ ]:
raw = load_raw(paths, cfg)
clean_df, cleaning_report = clean(raw, cfg)

display(cleaning_report.to_frame())
display(cleaning_report.missing_frame().assign(
    missing_share_before=lambda t: (t["missing_before"] / len(raw)).round(3)))

check("raw rows after removing duplicate grid cells", len(raw), 164_360)
check("locations", raw["location"].nunique(), 10)
check("columns with missing values",
      sorted(cleaning_report.missing_frame().index), ["ALLSKY_SFC_SW_DWN", "CLOUD_AMT"])
check("CLOUD_AMT missing share", round(raw["CLOUD_AMT"].isna().mean(), 3), 0.067, 0.001)
check("missing values after cleaning", int(clean_df.isna().sum().sum()), 0)
print("Date range:", cleaning_report.date_range[0].date(), "to",
      cleaning_report.date_range[1].date())

## 4. Creating the target

**What.** For each location, tomorrow's rainfall is attached to today's row and
converted to the binary label; the last day of each location, which has no tomorrow, is
dropped. Tomorrow's amount is kept as `tomorrow_mm` for error analysis only and is
explicitly excluded from every feature list.
**Why.** Shifting within each location prevents the last day of one district from being
paired with the first day of the next.
**How to read.** The share of rain-tomorrow days shows how imbalanced the classes are.

In [ ]:
labelled = add_target(clean_df, cfg)

check("rows after 1984 filter and target creation", len(labelled), 153_400)
check("share of RainTomorrow = 1", round(labelled[TARGET].mean(), 3), 0.593, 0.001)
labelled[TARGET].value_counts().rename(index={0: "no rain tomorrow", 1: "rain tomorrow"})

> **Interpretation (write in your own words):** ...
*(What does the class balance imply for the choice of metric and for a majority-class
predictor?)*

## 5. Exploratory analysis

**What.** Three views of the target: overall class balance, the rain rate by calendar
month and the rain rate by location.
**Why.** Rwanda has two rainy seasons, the long rains (March to May) and the short rains
(September to December), separated by drier periods; the monthly chart should show them
clearly. Differences between locations show how much the location itself explains.
Only the development period (training and validation years) is used, so the test years
remain completely unseen until the final evaluation.
**How to read.** Blue bars in the monthly chart mark the configured rainy-season months;
the dashed line is the overall rate.

In [ ]:
development = labelled[labelled["date"] <= cfg["split"]["val_end"]]

plots.plot_class_balance(development, TARGET, figures_dir=FIGURES)
plots.plot_monthly_rain_rate(development, TARGET, cfg["features"]["rainy_season_months"],
                             figures_dir=FIGURES)
plots.plot_rain_rate_by_location(development, TARGET, figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
*(Are the two rainy seasons visible? Which locations are wettest and why might that be,
for example altitude or proximity to Lake Kivu?)*

## 6. Feature engineering

**What.** `build_features` adds per-location history features and same-day regional
features, producing one table used by every model. Every feature uses only information
available at the end of the current day.

| Group | Features | Why |
|---|---|---|
| Raw parameters | 13 NASA parameters, latitude, longitude | Today's state of the atmosphere and soil |
| Rain history | `rain_today`, `rain_lag1..3`, `rain_sum_3d/7d/14d/30d`, `rain_days_7d`, `days_since_rain` | Wet and dry spells persist |
| Changes | `PS_change`, `RH2M_change`, `QV2M_change`, `T2M_change`, `CLOUD_AMT_change` | Falling pressure and rising humidity signal approaching systems |
| Derived | `temp_range`, `dewpoint_depression` | How close the air is to saturation, how cloudy the day was |
| Season | `doy_sin`, `doy_cos`, `rainy_season` | Circular day of year and the two rainy seasons |
| Wind | `wind_dir_sin`, `wind_dir_cos` (replace `WD2M`) | Direction is circular: 359 and 1 degree are neighbours |
| Regional (52-feature set only) | `rain_at_<Location>`, `region_RH2M`, `region_QV2M`, `region_CLOUD_AMT`, `region_PS_change`, `region_rain_share` | Rain systems move across the country; neighbours today inform here tomorrow |

**How to read.** The checks confirm the expected number of rows (the first three days
of each location are lost to the lags) and features.

In [ ]:
features_df, feature_sets = build_features(labelled, cfg)

check("rows after feature engineering", len(features_df), 153_370)
check("base numeric features", len(feature_sets.base), 37)
check("regional numeric features", len(feature_sets.regional), 52)
check("tomorrow_mm excluded from features", "tomorrow_mm" in feature_sets.regional, False)
features_df[feature_sets.regional].describe().T.round(2)

**What.** The features with the strongest linear correlation with the target, computed
on the training years only.
**How to read.** Blue bars raise the chance of rain tomorrow, orange bars lower it.
Correlation only captures one-feature, linear relationships, so it is a first look rather
than a selection rule.

In [ ]:
training_period = features_df[features_df["date"] <= cfg["split"]["train_end"]]
correlations = target_correlations(training_period, feature_sets.regional, TARGET,
                                   top=cfg["evaluation"]["top_features"])
plots.plot_top_correlations(correlations, figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
*(Which feature groups dominate? Are the strongest signals physically plausible?)*

## 7. Chronological split and baselines

**What.** The table is cut by date: training 1984 to 2015, validation 2016 to 2020, test
2021 to 2025.
**Why.** Consecutive days are strongly correlated and all locations share the same
weather on a given day. A random split would put the day before (or the same day at a
neighbouring site) of almost every test day into training, inflating every score. A
chronological split measures what matters: performance on future, unseen years.
Models are compared on validation; the test split is used once, at the very end.

In [ ]:
splits = split_by_year(features_df, cfg)
display(splits.summary())

check("train rows", len(splits.train), 116_850)
check("validation rows", len(splits.val), 18_270)
check("test rows", len(splits.test), 18_250)
for name, expected in (("train", 0.595), ("val", 0.596), ("test", 0.575)):
    check(f"{name} rain share", round(splits.part(name)[TARGET].mean(), 3), expected, 0.001)

**What.** Three rules that need no learning, scored on validation: always predict rain,
persistence (tomorrow equals today) and monthly climatology (the training-period rain
rate of the month, thresholded at one half).
**Why.** A model is only useful if it beats these. Persistence is the classic strong
weather baseline.
**How to read.** Macro F1 averages the F1 of both classes; always predicting rain scores
well on accuracy but poorly on macro F1 because it never predicts a dry day.

In [ ]:
baseline_rows = []
for spec in experiments["baselines"]:
    metrics = compute_metrics(splits.val[TARGET], predict_baseline(spec["kind"], splits, "val"))
    baseline_rows.append({"baseline": spec["name"], **metrics})
baseline_table = pd.DataFrame(baseline_rows).set_index("baseline").drop(columns="roc_auc")
display(baseline_table.round(3))

for run, expected in (("baseline_always_rain", 0.373), ("baseline_persistence", 0.787),
                      ("baseline_climatology", 0.702)):
    check(f"{run} validation macro F1", round(baseline_table.loc[run, "macro_f1"], 3),
          expected, 0.002)

> **Interpretation (write in your own words):** ...
*(How strong is persistence, and what does that say about how much skill a model can add?)*

## 8. Experiments

**What.** `run_experiments` executes every run listed in `configs/experiments.yaml`, in
the order they were performed:

* **Traditional ML (scikit-learn):** logistic regression, linear SVM, random forest and
  histogram gradient boosting with default settings; then randomised hyperparameter
  search for logistic regression, random forest and gradient boosting; then the two best
  tuned models retrained with the regional features.
* **Deep learning (Keras, `tf.data`):** a Sequential MLP improved step by step (batch
  normalisation, dropout, L2, learning rate, batch size, SGD with momentum, a deeper
  network), and a Functional API model with a learned location embedding.

**Why.** Changing one thing at a time makes the effect of each decision visible. Tuning
uses a `PredefinedSplit`, so every candidate is trained on the training years and scored
on the validation years, never on shuffled folds. The scaler and encoder are always
fitted on the training split only.
**Resuming.** With `run.mode: auto`, a run whose result and saved model are already on
Drive is reported as `[loaded]` and not retrained, so a disconnect costs nothing. Set
`mode: train` in `config.yaml` to retrain everything (about 40 minutes).

In [ ]:
log = ResultsLog.load(results_path(paths))
dl_data = DLData(splits, feature_sets, experiments["dl_defaults"]["feature_set"], paths, SEED)
log = run_experiments(experiments, splits, feature_sets, dl_data, log, paths, cfg)

**How to read.** One row per run, scored on the validation years. `Gap` is training
accuracy minus validation accuracy: a large gap signals overfitting. The second table
compares macro F1 with the values observed when the pipeline was validated (deep
learning varies slightly from run to run, so differences up to about 0.005 are normal).

In [ ]:
display(log.report_table())

REFERENCE_VAL_F1 = {
    "ml_01_logistic_regression": 0.802, "ml_02_linear_svm": 0.795,
    "ml_03_random_forest": 0.800, "ml_04_hist_gradient_boosting": 0.802,
    "ml_05_logreg_tuned": 0.803, "ml_06_random_forest_tuned": 0.801,
    "ml_07_hgb_tuned": 0.802, "ml_08_logreg_regional": 0.804, "ml_09_hgb_regional": 0.801,
    "dl_01_mlp_baseline": 0.797, "dl_02_batchnorm": 0.798, "dl_03_dropout_0.3": 0.801,
    "dl_04_l2_1e-4": 0.802, "dl_05_lr_3e-4": 0.798, "dl_06_batch_1024": 0.800,
    "dl_07_sgd_momentum": 0.800, "dl_08_deeper_256_128_64": 0.798,
    "dl_09_functional_embedding": 0.800,
}
reference = log.to_frame().set_index("run")[["macro_f1"]].join(
    pd.Series(REFERENCE_VAL_F1, name="reference"), how="inner")
reference["difference"] = reference["macro_f1"] - reference["reference"]
reference["status"] = np.where(reference["difference"].abs() <= 0.005, "OK", "CHECK")
reference.round(3)

**How to read.** Each dot is one run's validation macro F1, coloured by paradigm; the
dashed line is the persistence baseline. Differences between models are small, which is
why a dot plot with a narrow axis is used instead of bars.

In [ ]:
plots.plot_baselines_vs_models(log.to_frame(), cfg["evaluation"]["primary_metric"],
                               figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
**Traditional ML.** *(How much did tuning help each model? Which models overfit, judging
by the gap? Did the regional features help?)*

> **Interpretation (write in your own words):** ...
**Deep learning.** *(What did each change (batch normalisation, dropout, L2, learning
rate, batch size, optimiser, depth, embedding) do to validation macro F1 and to the gap?)*

## 9. Learning curves

**What.** The tuned logistic regression and the tuned gradient boosting are retrained on
random fractions of the training split (5% to 100%), and both training and validation
macro F1 are recorded. For the network, the training history (loss, accuracy, AUC per
epoch) of the best deep learning run is shown.
**Why.** Learning curves separate bias from variance. Curves that converge at a modest
score point to bias (more data will not help much); a persistent gap points to variance
(more data or more regularisation would help).
**How to read.** Blue is training, orange is validation; the dashed vertical line in the
history marks the epoch restored by early stopping. Results are cached in
`results/learning_curves_ml.csv`.

In [ ]:
best_params = load_best_params(paths.results_dir / "best_params.json")
ml_specs = {spec["name"]: spec for spec in experiments["ml"]}
curves = pd.concat([
    ml_learning_curve(ml_specs[run], splits, feature_sets, paths,
                      cfg["evaluation"]["learning_curve_fractions"],
                      best_params=best_params.get(run), seed=SEED)
    for run in ("ml_05_logreg_tuned", "ml_07_hgb_tuned")
], ignore_index=True)
display(curves.round(4))
plots.plot_ml_learning_curves(curves, figures_dir=FIGURES);

In [ ]:
best_dl_run = select_best(log, "dl", cfg["evaluation"]["primary_metric"], paths)
with open(paths.histories_dir / f"{best_dl_run}.json", encoding="utf-8") as handle:
    best_dl_history = json.load(handle)
plots.plot_dl_history(best_dl_history, best_dl_run, figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
*(Bias or variance? Would more years of data help either model? When did the network
stop improving?)*

## 10. Model selection

**What.** Within each paradigm, the run with the highest validation macro F1 is selected;
scores are compared to four decimals and ties go to the faster model. Runs that did not
save a model (the random forests, whose files are very large) cannot be selected.
**Why.** The rule is fixed in code before the test split is touched, so the final
comparison cannot be influenced by test results. The selected models are saved as
`best_ml_model.joblib`, `best_dl_model.keras` and `best_dl_model.h5`.

In [ ]:
metric = cfg["evaluation"]["primary_metric"]
columns = ["run", "feature_set", metric, "roc_auc", "gap", "seconds"]
display(rank_runs(log, "ml", metric)[columns].head(5).round(4))
display(rank_runs(log, "dl", metric)[columns].head(5).round(4))

best_ml = select_best(log, "ml", metric, paths)
best_dl = select_best(log, "dl", metric, paths)
check("selected ML model", best_ml, "ml_08_logreg_regional")
check("selected DL model", best_dl, "dl_04_l2_1e-4")

final_files = save_final_models(best_ml, best_dl, paths)
for name, path in final_files.items():
    print(f"{name}: {path}")

> **Justification (write in your own words):** ...
*(Why are these two models the right choices? Consider the margin over the runners-up,
the gap, training time and simplicity.)*

## 11. Final test evaluation

**What.** The two saved final models and the persistence baseline are evaluated on the
test years (2021 to 2025). This is the only cell in the project that scores the test
split. The metrics are written to `results/final_test_metrics.csv`; if that file already
exists it is not overwritten, so the official result cannot be replaced silently.
**How to read.** Compare each model with persistence and with its own validation score;
a large drop would indicate overfitting to the validation years during tuning.

In [ ]:
test_results = final_test_evaluation(best_ml, best_dl, splits, dl_data, paths,
                                     threshold=cfg["evaluation"]["threshold"])
display(test_results.metrics.round(3))

LABELS = {"best_ml": f"ML: {best_ml}", "best_dl": f"DL: {best_dl}",
          "persistence": "Persistence"}

> **Interpretation (write in your own words):** ...
*(Do the models beat persistence on unseen years? How do test scores compare with
validation scores? Note that the test period is slightly drier.)*

## 12. Confusion matrices and ROC curves

**What.** Confusion matrices of both final models on the test split, and ROC curves with
the persistence rule as a single operating point.
**How to read.** In the matrices, rows are the actual class and each cell shows the
count and the share of that row: the diagonal holds correct predictions, the top-right
cell false alarms and the bottom-left missed rain. On the ROC plot, a curve above the
persistence point means the model can beat persistence on both hit rate and false-alarm
rate at some threshold.

In [ ]:
y_test = test_results.y_true
plots.plot_confusion_matrices(
    y_test, {LABELS[k]: test_results.predictions[k] for k in ("best_ml", "best_dl")},
    figures_dir=FIGURES)
plots.plot_roc_curves(
    y_test, {LABELS[k]: test_results.scores[k] for k in ("best_ml", "best_dl")},
    points={LABELS["persistence"]: test_results.predictions["persistence"]},
    figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
*(Which error type dominates, false alarms or missed rain? Which matters more to a
farmer deciding whether to spray?)*

## 13. Error analysis

**What.** Test error rates broken down by calendar month, by tomorrow's rainfall amount,
by "change days" (tomorrow differs from today) versus persistent days, and by location,
plus the most confident false positives and false negatives of each final model.
**Why.** An aggregate score hides where models fail. The rainfall buckets show whether
errors concentrate near the 1 mm threshold, where the label is almost arbitrary; the
change-day table shows how much of the skill is just persistence.
**How to read.** Each table gives the error rate per model and the number of days.

In [ ]:
named_predictions = {LABELS[k]: v for k, v in test_results.predictions.items()}
named_scores = {LABELS[k]: v for k, v in test_results.scores.items()}
error_tables = error_analysis(splits.test, named_predictions,
                              bucket_edges=cfg["evaluation"]["error_bucket_edges_mm"],
                              scores=named_scores, top=cfg["evaluation"]["top_errors"],
                              target=TARGET)
for name in ("by_change", "by_bucket", "by_month", "by_location"):
    print(name)
    display(error_tables[name].round(3))

In [ ]:
model_labels = list(named_predictions)
plots.plot_error_by_bucket(error_tables["by_bucket"], model_labels, figures_dir=FIGURES)
plots.plot_error_by_month(error_tables["by_month"], model_labels, figures_dir=FIGURES);

**What.** The days on which each final model was most confidently wrong.
**How to read.** `score` is the predicted probability of rain; false positives are
sorted from the most confident, false negatives from the least likely according to the
model. Look for shared patterns (season, location, rainfall just below 1 mm).

In [ ]:
for name, table in error_tables.items():
    if name.startswith("false_"):
        print(name)
        display(table.round(3))

> **Interpretation (write in your own words):** ...
*(Where and why do the models fail? Are the errors concentrated on change days, near the
threshold, in particular months or locations?)*

## 14. Feature importance

**What.** For the logistic regression, the largest standardised coefficients (the change
in log-odds of rain tomorrow for a one standard deviation increase in the feature). For
the network, permutation importance on the test split: the drop in macro F1 when one
feature is shuffled.
**Why.** Comparing what each paradigm relies on tests whether they learned the same
physical signals. Permutation importance is needed for the network, which has no
readable coefficients.
**How to read.** Blue coefficients raise the probability of rain, orange lower it. For
permutation importance, a larger drop means a more important feature; values near zero
mean the network barely uses the feature (or another correlated feature replaces it).

In [ ]:
ml_model = joblib.load(final_files["ml"])
dl_model = keras.models.load_model(final_files["dl_keras"])
top = cfg["evaluation"]["top_features"]

if hasattr(ml_model.named_steps["model"], "coef_"):
    coefficients = logreg_coefficients(ml_model, top=top)
    display(coefficients.round(3))
    plots.plot_feature_importance(coefficients, "coefficient",
                                  f"Largest coefficients of {best_ml}",
                                  "importance_ml_coefficients",
                                  xlabel="Standardised coefficient (log-odds)",
                                  figures_dir=FIGURES)
else:
    print(f"{best_ml} is not a linear model; coefficients are not available.")

In [ ]:
permutation = dl_permutation_importance(dl_model, dl_data,
                                        threshold=cfg["evaluation"]["threshold"], seed=SEED)
display(permutation.head(top).round(4))
plots.plot_feature_importance(permutation.head(top), "macro_f1_drop",
                              f"Permutation importance of {best_dl} (test split)",
                              "importance_dl_permutation",
                              xlabel="Drop in macro F1 when shuffled",
                              figures_dir=FIGURES);

> **Interpretation (write in your own words):** ...
*(Do both models rely on the same features? Are the most important features physically
meaningful?)*

## 15. Traditional ML versus deep learning

**What.** The two final models side by side: validation and test macro F1 and ROC-AUC,
training time (of the final fit, excluding the hyperparameter search), number of
learned parameters and interpretability.

In [ ]:
comparison = paradigm_comparison(log, test_results.metrics,
                                 {"best_ml": ml_model, "best_dl": dl_model})
comparison.round(3)

> **Discussion (write in your own words):** ...
*(Which paradigm would you deploy for Rwandan farmers, and why? Consider accuracy,
cost, interpretability and the literature on tree models and neural networks for tabular
data.)*

## 16. Limitations

> **Limitations (write in your own words):** ...
*(Points to consider: NASA POWER values are reanalysis and satellite estimates for grid
cells, not rain-gauge measurements; the daily resolution hides the timing of rain within
the day; the 1 mm target is derived and borderline days are almost arbitrary; the test
period is slightly drier than training; only 10 locations.)*

## 17. Conclusion

> **Conclusion (write in your own words):** ...

## 18. References

* NASA Langley Research Center. *POWER Project: Prediction Of Worldwide Energy Resources,
  Daily API documentation.* https://power.larc.nasa.gov/docs/services/api/temporal/daily/
* Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5 to 32.
* Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine.
  *Annals of Statistics*, 29(5), 1189 to 1232.
* Ioffe, S., and Szegedy, C. (2015). Batch normalization: Accelerating deep network
  training by reducing internal covariate shift. *Proceedings of ICML*, 448 to 456.
* Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., and Salakhutdinov, R.
  (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of
  Machine Learning Research*, 15, 1929 to 1958.
* Grinsztajn, L., Oyallon, E., and Varoquaux, G. (2022). Why do tree-based models still
  outperform deep learning on typical tabular data? *Advances in Neural Information
  Processing Systems 35 (Datasets and Benchmarks Track)*.